In [9]:
!pip install folium plotly pandas requests ipywidgets --quiet

In [12]:
# ============================================================
# CELL 1 — Install
# ============================================================
!pip install folium plotly pandas ipywidgets --quiet

# ============================================================
# CELL 2 — Imports
# ============================================================
import pandas as pd
import numpy as np
import io
import folium
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

# ============================================================
# CELL 3 — Upload CSVs Manually
# Download from: https://catalog.data.gov/dataset/active-transportation-count-dataset
# Upload pedestrian + bicycle CSVs (any year)
# ============================================================
# ============================================================
# CELL 3 — Load CSVs Directly (already uploaded to Colab)
# ============================================================
FILES = {
    '/content/caltrans_bicycle_hourly_count_2024.csv':    ('Bicycle',    '2024'),
    '/content/caltrans_bicycle_hourly_count_2025.csv':    ('Bicycle',    '2025'),
    '/content/caltrans_pedestrian_hourly_count_2024.csv': ('Pedestrian', '2024'),
    '/content/caltrans_pedestrian_hourly_count_2025.csv': ('Pedestrian', '2025'),
}

dfs = []
for filepath, (mode, year) in FILES.items():
    chunk = pd.read_csv(filepath, low_memory=False)
    chunk['mode'] = mode
    chunk['year'] = year
    dfs.append(chunk)
    print(f'✅ {filepath.split("/")[-1]}: {len(chunk):,} rows | columns: {list(chunk.columns)}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\n✅ Total rows: {len(df_raw):,}')
df_raw.head(3)

# ============================================================
# CELL 4 — Clean & Standardize
# ============================================================
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

# Normalize column name variants
rename_map = {
    'lat': 'latitude', 'lng': 'longitude', 'lon': 'longitude',
    'volume': 'count', 'counts': 'count',
    'time_hour': 'hour',
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

# Extract hour from datetime if 'hour' column is missing
if 'hour' not in df.columns:
    for col in ['date', 'datetime', 'timestamp', 'date_hour']:
        if col in df.columns:
            df['hour'] = pd.to_datetime(df[col], errors='coerce').dt.hour
            print(f'   ℹ️  Extracted hour from "{col}"')
            break

for col in ['latitude', 'longitude', 'count', 'hour']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['latitude', 'longitude', 'count', 'hour'], inplace=True)
df = df[df['latitude'].between(32, 42) & df['longitude'].between(-125, -114)]
df['hour']  = df['hour'].astype(int)
df['count'] = df['count'].clip(lower=0)

print(f'✅ Clean: {len(df):,} rows')
print(f'   Hours: {df["hour"].min()}–{df["hour"].max()}')
print(f'   Modes: {df["mode"].value_counts().to_dict()}')
print(f'   Years: {df["year"].value_counts().to_dict()}')

# ============================================================
# CELL 5 — Static Folium Map (all hours combined)
# ============================================================
agg = (
    df.groupby(['latitude', 'longitude'])
    .apply(lambda g: pd.Series({
        'total':       g['count'].sum(),
        'pedestrians': g.loc[g['mode'] == 'Pedestrian', 'count'].sum(),
        'cyclists':    g.loc[g['mode'] == 'Bicycle',    'count'].sum(),
    }))
    .reset_index()
)

m = folium.Map(
    location=[agg['latitude'].mean(), agg['longitude'].mean()],
    zoom_start=6, tiles='CartoDB positron'
)
max_count = agg['total'].max() or 1
for _, row in agg.iterrows():
    radius   = 5 + (row['total'] / max_count) * 25
    pct_bike = row['cyclists'] / (row['total'] or 1)
    color    = '#3B82F6' if pct_bike > 0.6 else ('#E85D26' if pct_bike < 0.3 else '#8B5CF6')
    folium.CircleMarker(
        [row['latitude'], row['longitude']],
        radius=radius, color=color, fill=True, fill_opacity=0.7,
        popup=folium.Popup(
            f"Total: {int(row['total']):,}<br>"
            f"Pedestrians: {int(row['pedestrians']):,}<br>"
            f"Cyclists: {int(row['cyclists']):,}",
            max_width=220
        ),
        tooltip=f"Total: {int(row['total']):,}"
    ).add_to(m)

legend = '''<div style="position:fixed;bottom:30px;left:30px;background:white;
     padding:12px 16px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.2);
     font-family:Arial;font-size:13px;z-index:1000;">
  <b>Active Transportation</b><br>
  <span style="color:#E85D26">●</span> Mostly Pedestrian<br>
  <span style="color:#8B5CF6">●</span> Mixed<br>
  <span style="color:#3B82F6">●</span> Mostly Bicycle<br>
  <small>Bubble size = total count</small></div>'''
m.get_root().html.add_child(folium.Element(legend))
m.save('/content/static_map.html')
print('✅ Static map saved → /content/static_map.html')
display(m)

# ============================================================
# CELL 6 — Animated Plotly Map (time-of-day slider)
# ============================================================
hourly = (
    df.groupby(['latitude', 'longitude', 'hour'])
    .apply(lambda g: pd.Series({
        'total':       g['count'].sum(),
        'pedestrians': g.loc[g['mode'] == 'Pedestrian', 'count'].sum(),
        'cyclists':    g.loc[g['mode'] == 'Bicycle',    'count'].sum(),
    }))
    .reset_index()
)
hourly['size']       = np.sqrt(hourly['total'] + 1) * 3
hourly['hour_label'] = hourly['hour'].apply(lambda h: f"{h%12 or 12}{'am' if h<12 else 'pm'}")
hourly = hourly.sort_values('hour')

fig = px.scatter_mapbox(
    hourly, lat='latitude', lon='longitude',
    size='size', color='cyclists',
    color_continuous_scale=['#E85D26', '#8B5CF6', '#3B82F6'],
    animation_frame='hour',
    hover_data={
        'total': True, 'pedestrians': True, 'cyclists': True,
        'hour_label': True, 'latitude': False, 'longitude': False, 'size': False,
    },
    labels={'cyclists': 'Cyclists', 'total': 'Total', 'pedestrians': 'Pedestrians', 'hour_label': 'Hour'},
    mapbox_style='carto-positron', zoom=5,
    center={'lat': df['latitude'].mean(), 'lon': df['longitude'].mean()},
    title='🚶🚲 Caltrans Active Transportation — by Location & Hour',
    height=640,
)
fig.update_layout(
    coloraxis_colorbar=dict(title='← Ped | Bike →', thickness=14, len=0.6),
    margin=dict(t=50, b=10, l=10, r=10),
    updatemenus=[{
        'type': 'buttons', 'showactive': False, 'y': 0, 'x': 0.5, 'xanchor': 'center',
        'buttons': [
            {'label': '▶ Play',  'method': 'animate',
             'args': [None, {'frame': {'duration': 700, 'redraw': True}, 'fromcurrent': True}]},
            {'label': '⏸ Pause', 'method': 'animate',
             'args': [[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]},
        ]
    }]
)
fig.show()
fig.write_html('/content/animated_map.html')
print('✅ Animated map saved → /content/animated_map.html')
print('💡 Drag the hour slider or press ▶ Play')

# ============================================================
# CELL 7 — 24-Hour Polar Dial
# ============================================================
# ============================================================
# CELL 7 — 24-Hour Polar Dial (fixed fillcolor)
# ============================================================
pivot = (
    df.groupby(['hour', 'mode'])['count'].sum().reset_index()
    .pivot_table(index='hour', columns='mode', values='count', fill_value=0)
    .reindex(range(24), fill_value=0)
)
hour_labels = [f"{h%12 or 12}{'am' if h<12 else 'pm'}" for h in range(24)]

fig2 = go.Figure()
for mode, color, rgba_fill in [
    ('Pedestrian', '#E85D26', 'rgba(232,93,38,0.2)'),
    ('Bicycle',    '#3B82F6', 'rgba(59,130,246,0.2)'),
]:
    if mode in pivot.columns:
        vals = list(pivot[mode]) + [pivot[mode].iloc[0]]
        fig2.add_trace(go.Scatterpolar(
            r=vals, theta=hour_labels + [hour_labels[0]],
            fill='toself', fillcolor=rgba_fill,
            line=dict(color=color, width=2), name=mode,
            hovertemplate='%{theta}: %{r:,.0f}<extra>%{fullData.name}</extra>'
        ))
fig2.update_layout(
    polar=dict(
        angularaxis=dict(tickmode='array', tickvals=hour_labels,
                         direction='clockwise', rotation=90),
        radialaxis=dict(tickfont=dict(size=9)),
    ),
    title=dict(text='🕐 Pedestrian & Bicycle Counts by Hour of Day', x=0.5),
    height=540, font=dict(family='Arial', size=13),
)
fig2.show()
fig2.write_html('/content/time_dial.html')
print('✅ Polar dial saved')

# ============================================================
# CELL 8 — Interactive Hour Slider (ipywidgets + Folium)
# ============================================================
def plot_hour_map(hour):
    filtered = df[df['hour'] == hour]
    agg_h = (
        filtered.groupby(['latitude', 'longitude'])
        .apply(lambda g: pd.Series({
            'total':       g['count'].sum(),
            'pedestrians': g.loc[g['mode'] == 'Pedestrian', 'count'].sum(),
            'cyclists':    g.loc[g['mode'] == 'Bicycle',    'count'].sum(),
        }))
        .reset_index()
    )
    m2 = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()],
                    zoom_start=6, tiles='CartoDB positron')
    max_c = agg_h['total'].max() or 1
    for _, row in agg_h.iterrows():
        r     = 4 + (row['total'] / max_c) * 22
        color = '#3B82F6' if row['cyclists'] > row['pedestrians'] else '#E85D26'
        folium.CircleMarker(
            [row['latitude'], row['longitude']], radius=r,
            color=color, fill=True, fill_opacity=0.7,
            tooltip=f"Total:{int(row['total'])} | Ped:{int(row['pedestrians'])} | Bike:{int(row['cyclists'])}"
        ).add_to(m2)
    hour_str = f"{hour%12 or 12}{'am' if hour<12 else 'pm'}"
    display(HTML(f'<h3 style="font-family:Arial">🕐 Hour: {hour_str} | Sites: {len(agg_h)}</h3>'))
    display(m2)

slider = widgets.IntSlider(
    value=8, min=0, max=23, step=1,
    description='Hour (0–23):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
widgets.interact(plot_hour_map, hour=slider);

# ============================================================
# CELL 9 — Download All Outputs
# ============================================================
from google.colab import files
for path in ['/content/static_map.html', '/content/animated_map.html', '/content/time_dial.html']:
    files.download(path)
print('✅ All files downloaded!')
# ============================================================
# SAVE TO GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Create a folder in your Drive
save_folder = '/content/drive/My Drive/Active Transportation Maps'
os.makedirs(save_folder, exist_ok=True)

# Copy all HTML visualizations
for filename in ['static_map.html', 'animated_map.html', 'time_dial.html']:
    src = f'/content/{filename}'
    dst = f'{save_folder}/{filename}'
    shutil.copy(src, dst)
    print(f'✅ Saved → {dst}')

print('\n📂 Open Google Drive → "Active Transportation Maps" folder')
print('   Double-click any .html file → Open with Google Chrome')

✅ Libraries loaded
✅ caltrans_bicycle_hourly_count_2024.csv: 605,930 rows | columns: ['loc_id', 'district', 'latitude', 'longitude', 'date', 'mode', 'direction', 'count', 'count_type', 'year']
✅ caltrans_bicycle_hourly_count_2025.csv: 782,357 rows | columns: ['loc_id', 'district', 'latitude', 'longitude', 'date', 'mode', 'direction', 'count', 'count_type', 'year']
✅ caltrans_pedestrian_hourly_count_2024.csv: 1,158,488 rows | columns: ['loc_id', 'district', 'latitude', 'longitude', 'date', 'mode', 'direction', 'count', 'count_type', 'year']
✅ caltrans_pedestrian_hourly_count_2025.csv: 713,750 rows | columns: ['loc_id', 'district', 'latitude', 'longitude', 'date', 'mode', 'direction', 'count', 'count_type', 'year']

✅ Total rows: 3,260,525
   ℹ️  Extracted hour from "date"
✅ Clean: 3,260,523 rows
   Hours: 0–23
   Modes: {'Pedestrian': 1872237, 'Bicycle': 1388286}
   Years: {'2024': 1764418, '2025': 1496105}
✅ Static map saved → /content/static_map.html


✅ Animated map saved → /content/animated_map.html
💡 Drag the hour slider or press ▶ Play


✅ Polar dial saved


interactive(children=(IntSlider(value=8, description='Hour (0–23):', layout=Layout(width='500px'), max=23, sty…

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
Mounted at /content/drive
✅ Saved → /content/drive/My Drive/Active Transportation Maps/static_map.html
✅ Saved → /content/drive/My Drive/Active Transportation Maps/animated_map.html
✅ Saved → /content/drive/My Drive/Active Transportation Maps/time_dial.html

📂 Open Google Drive → "Active Transportation Maps" folder
   Double-click any .html file → Open with Google Chrome
